# PEC2 - Procesamiento del Lenguaje Natural

En esta práctica se trabaja con AG News para resolver una tarea de clasificación automática de noticias en 4 categorías. La idea no es solo entrenar modelos y sacar métricas, sino comparar enfoques distintos y ver qué gana y qué pierde cada uno.

El cuaderno está organizado en cuatro partes: análisis del conjunto de datos, modelos recurrentes, modelos Transformer y una pequeña parte final de explicabilidad con mapas de atención. 

El objetivo es terminar con una comparación razonada entre los modelos y no quedarse solo en los números.

In [ ]:
import os
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

import re
import gc
import random
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report
)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset, Dataset as HFDataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    set_seed
)

warnings.filterwarnings("ignore")

SEED = 11
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo detectado:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

HF_CACHE_DIR = os.path.abspath("./hf_cache")
os.makedirs(HF_CACHE_DIR, exist_ok=True)
print("Cache HF:", HF_CACHE_DIR)

In [ ]:
VAL_SIZE = 0.10
NUM_CLASSES = 4
LABEL_NAMES = ["World", "Sports", "Business", "Sci/Tech"]


MAX_VOCAB = 50000
EMBED_DIM = 200
HIDDEN_DIM = 256
DROPOUT = 0.30
RNN_BATCH_SIZE = 256 if torch.cuda.is_available() else 128
RNN_EPOCHS = 10
RNN_LR = 1e-3
RNN_PATIENCE = 2


TRANSFORMER_MAX_LEN = 128
DISTIL_BATCH = 16 if torch.cuda.is_available() else 8
BERT_BATCH = 8 if torch.cuda.is_available() else 4
TRANSFORMER_EPOCHS = 3
TRANSFORMER_LR = 2e-5
TRANSFORMER_WEIGHT_DECAY = 0.01
TRANSFORMER_WARMUP_RATIO = 0.1
TRANSFORMER_PATIENCE = 1

MODEL_3_NAME = "distilbert-base-uncased"
MODEL_4_NAME = "bert-base-uncased"

## 1. Carga del dataset y creación de train, validación y test

Se utilizará AG News. El conjunto ya viene separado en entrenamiento y prueba, así que la validación se obtiene a partir del entrenamiento original. Así, la validación sirve para ajustar y comparar modelos, mientras que el test queda reservado para la evaluación final.

In [ ]:
raw = load_dataset("ag_news")
raw

In [ ]:
train_df = raw["train"].to_pandas()
test_df = raw["test"].to_pandas()

def build_text_column(df: pd.DataFrame) -> pd.Series:
    if {"title", "description"}.issubset(df.columns):
        text = (
            df["title"].fillna("").astype(str).str.strip() + ". " +
            df["description"].fillna("").astype(str).str.strip()
        )
    elif "text" in df.columns:
        text = df["text"].fillna("").astype(str).str.strip()
    else:
        raise ValueError(f"No encuentro una columna de texto válida. Columnas disponibles: {df.columns.tolist()}")
    return text.str.replace(r"\s+", " ", regex=True).str.strip()

train_df["text"] = build_text_column(train_df)
test_df["text"] = build_text_column(test_df)

train_df = train_df[["text", "label"]].copy()
test_df = test_df[["text", "label"]].copy()

train_df, val_df = train_test_split(
    train_df,
    test_size=VAL_SIZE,
    random_state=SEED,
    stratify=train_df["label"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Test :", test_df.shape)

## 2. Análisis exploratorio del texto

Antes de entrenar los modelos es mejor revisar dos cosas: si las clases están equilibradas y cómo son de largos los textos. Esto ayuda a justificar el preprocesado y sobre todo, a decidir una longitud de secuencia razonable sin recortar demasiado.

In [ ]:
def add_length_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["n_chars"] = out["text"].str.len()
    out["n_words"] = out["text"].str.split().str.len()
    return out

train_eda = add_length_features(train_df)
val_eda = add_length_features(val_df)
test_eda = add_length_features(test_df)

display(train_eda.head())
print(train_eda["n_words"].describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.countplot(x=train_df["label"].map(dict(enumerate(LABEL_NAMES))), ax=axes[0])
axes[0].set_title("Distribución de clases (train)")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=20)

sns.histplot(train_eda["n_words"], bins=60, ax=axes[1])
axes[1].set_title("Distribución de longitud en palabras")
axes[1].set_xlabel("Número de palabras")

plt.tight_layout()
plt.show()

print("Distribución de clases en train:")
display(train_df["label"].map(dict(enumerate(LABEL_NAMES))).value_counts().sort_index())

In [ ]:
def quick_clean(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

def top_words(series, n=20):
    counter = Counter()
    for text in series:
        tokens = re.findall(r"[a-zA-Z0-9']+", quick_clean(text))
        counter.update(tokens)
    return pd.DataFrame(counter.most_common(n), columns=["token", "freq"])

display(top_words(train_df["text"], n=20))

## 3. Preprocesado para los modelos recurrentes

Para GRU y LSTM se utiliza una tokenización sencilla a nivel de palabra. En este caso sí tiene sentido construir un vocabulario propio a partir del conjunto de entrenamiento, porque estos modelos no vienen con un tokenizador preentrenado.

El texto se pasa a minúsculas y se limpia de forma moderada. La idea es conservar bastante información del texto original sin complicar demasiado el pipeline.

In [ ]:
def clean_text_for_rnn(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9\s']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize_words(text: str):
    return re.findall(r"[a-z0-9']+", clean_text_for_rnn(text))

train_tokens = [tokenize_words(t) for t in tqdm(train_df["text"], desc="Tokenizando train")]
val_tokens = [tokenize_words(t) for t in tqdm(val_df["text"], desc="Tokenizando val")]
test_tokens = [tokenize_words(t) for t in tqdm(test_df["text"], desc="Tokenizando test")]

In [ ]:
counter = Counter()
for tokens in train_tokens:
    counter.update(tokens)

special_tokens = ["<pad>", "<unk>"]
vocab_words = [w for w, _ in counter.most_common(MAX_VOCAB - len(special_tokens))]
itos = special_tokens + vocab_words
stoi = {w: i for i, w in enumerate(itos)}

PAD_IDX = stoi["<pad>"]
UNK_IDX = stoi["<unk>"]

train_token_lens = [len(x) for x in train_tokens]
MAX_LEN_RNN = max(32, min(128, int(np.percentile(train_token_lens, 95))))

print("Tamaño del vocabulario:", len(stoi))
print("Longitud máxima elegida para GRU y LSTM:", MAX_LEN_RNN)

In [ ]:
class NewsSequenceDataset(Dataset):
    def __init__(self, tokens_list, labels, stoi, max_len):
        self.labels = labels.tolist()
        self.stoi = stoi
        self.max_len = max_len
        self.sequences = [self.encode(tokens) for tokens in tokens_list]

    def encode(self, tokens):
        ids = [self.stoi.get(tok, UNK_IDX) for tok in tokens][:self.max_len]
        length = len(ids)
        if length < self.max_len:
            ids = ids + [PAD_IDX] * (self.max_len - length)
        return torch.tensor(ids, dtype=torch.long), torch.tensor(length, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        ids, length = self.sequences[idx]
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return ids, length, label

train_rnn_ds = NewsSequenceDataset(train_tokens, train_df["label"], stoi, MAX_LEN_RNN)
val_rnn_ds = NewsSequenceDataset(val_tokens, val_df["label"], stoi, MAX_LEN_RNN)
test_rnn_ds = NewsSequenceDataset(test_tokens, test_df["label"], stoi, MAX_LEN_RNN)

train_rnn_loader = DataLoader(train_rnn_ds, batch_size=RNN_BATCH_SIZE, shuffle=True, num_workers=0)
val_rnn_loader = DataLoader(val_rnn_ds, batch_size=RNN_BATCH_SIZE, shuffle=False, num_workers=0)
test_rnn_loader = DataLoader(test_rnn_ds, batch_size=RNN_BATCH_SIZE, shuffle=False, num_workers=0)

## 4. Funciones comunes para GRU y LSTM

In [ ]:
class RecurrentClassifier(nn.Module):
    def __init__(
        self,
        vocab_size,
        embed_dim=200,
        hidden_dim=256,
        num_layers=1,
        dropout=0.3,
        num_classes=4,
        cell_type="gru",
        bidirectional=True
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.dropout = nn.Dropout(dropout)
        self.bidirectional = bidirectional
        self.cell_type = cell_type.lower()

        rnn_dropout = dropout if num_layers > 1 else 0.0

        if self.cell_type == "gru":
            self.rnn = nn.GRU(
                embed_dim,
                hidden_dim,
                num_layers=num_layers,
                batch_first=True,
                dropout=rnn_dropout,
                bidirectional=bidirectional
            )
        elif self.cell_type == "lstm":
            self.rnn = nn.LSTM(
                embed_dim,
                hidden_dim,
                num_layers=num_layers,
                batch_first=True,
                dropout=rnn_dropout,
                bidirectional=bidirectional
            )
        else:
            raise ValueError("cell_type debe ser 'gru' o 'lstm'.")

        out_dim = hidden_dim * 2 if bidirectional else hidden_dim
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(out_dim, num_classes)
        )

    def forward(self, input_ids, lengths):
        x = self.embedding(input_ids)
        x = self.dropout(x)

        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, hidden = self.rnn(packed)

        if self.cell_type == "lstm":
            hidden = hidden[0]

        if self.bidirectional:
            final_hidden = torch.cat([hidden[-2], hidden[-1]], dim=1)
        else:
            final_hidden = hidden[-1]

        logits = self.classifier(final_hidden)
        return logits

In [ ]:
def compute_metrics_np(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    return {
        "accuracy": acc,
        "precision_macro": prec,
        "recall_macro": rec,
        "f1_macro": f1
    }

def evaluate_recurrent(model, dataloader, criterion):
    model.eval()
    losses = []
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for input_ids, lengths, labels in dataloader:
            input_ids = input_ids.to(device)
            lengths = lengths.to(device)
            labels = labels.to(device)

            logits = model(input_ids, lengths)
            loss = criterion(logits, labels)

            losses.append(loss.item())
            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    metrics = compute_metrics_np(all_labels, all_preds)
    metrics["loss"] = float(np.mean(losses))
    return metrics, np.array(all_labels), np.array(all_preds)

def train_recurrent_model(model, train_loader, val_loader, epochs=10, lr=1e-3, patience=2):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = {
        "train_loss": [],
        "val_loss": [],
        "val_accuracy": [],
        "val_f1_macro": []
    }

    best_state = None
    best_val_f1 = -1
    patience_counter = 0

    model.to(device)

    for epoch in range(1, epochs + 1):
        model.train()
        train_losses = []

        for input_ids, lengths, labels in tqdm(train_loader, desc=f"Epoch {epoch}/{epochs}", leave=False):
            input_ids = input_ids.to(device)
            lengths = lengths.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            logits = model(input_ids, lengths)
            loss = criterion(logits, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            train_losses.append(loss.item())

        val_metrics, _, _ = evaluate_recurrent(model, val_loader, criterion)

        history["train_loss"].append(float(np.mean(train_losses)))
        history["val_loss"].append(val_metrics["loss"])
        history["val_accuracy"].append(val_metrics["accuracy"])
        history["val_f1_macro"].append(val_metrics["f1_macro"])

        print(
            f"Epoch {epoch:02d} | "
            f"train_loss={history['train_loss'][-1]:.4f} | "
            f"val_loss={val_metrics['loss']:.4f} | "
            f"val_acc={val_metrics['accuracy']:.4f} | "
            f"val_f1={val_metrics['f1_macro']:.4f}"
        )

        if val_metrics["f1_macro"] > best_val_f1:
            best_val_f1 = val_metrics["f1_macro"]
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("Early stopping activado.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, history

def plot_learning_curves(history, title):
    epochs = range(1, len(history["train_loss"]) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(epochs, history["train_loss"], label="train_loss")
    axes[0].plot(epochs, history["val_loss"], label="val_loss")
    axes[0].set_title(f"{title} - Pérdida")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    axes[1].plot(epochs, history["val_accuracy"], label="val_accuracy")
    axes[1].plot(epochs, history["val_f1_macro"], label="val_f1_macro")
    axes[1].set_title(f"{title} - Validación")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

## 5. Modelo 1 - GRU

Como primer modelo se entrena una GRU bidireccional. La tomo como punto de partida porque suele funcionar razonablemente bien en clasificación de texto y, además, es algo más ligera que una LSTM.

In [ ]:
gru_model = RecurrentClassifier(
    vocab_size=len(stoi),
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=1,
    dropout=DROPOUT,
    num_classes=NUM_CLASSES,
    cell_type="gru",
    bidirectional=True
)

gru_model, gru_history = train_recurrent_model(
    gru_model,
    train_rnn_loader,
    val_rnn_loader,
    epochs=RNN_EPOCHS,
    lr=RNN_LR,
    patience=RNN_PATIENCE
)

plot_learning_curves(gru_history, "GRU")

In [ ]:
criterion = nn.CrossEntropyLoss()
gru_test_metrics, y_true_gru, y_pred_gru = evaluate_recurrent(gru_model, test_rnn_loader, criterion)

print("Métricas en test - GRU")
for k, v in gru_test_metrics.items():
    print(f"{k}: {v:.4f}")

print("\nClassification report - GRU")
print(classification_report(y_true_gru, y_pred_gru, target_names=LABEL_NAMES, digits=4))

## 6. Modelo 2 - LSTM

A continuación se entrena una LSTM con una estructura parecida a la de la GRU. La comparación interesa porque ambas redes son recurrentes, pero la LSTM incorpora una memoria algo más rica y eso puede notarse en la clasificación.

In [ ]:
lstm_model = RecurrentClassifier(
    vocab_size=len(stoi),
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=1,
    dropout=DROPOUT,
    num_classes=NUM_CLASSES,
    cell_type="lstm",
    bidirectional=True
)

lstm_model, lstm_history = train_recurrent_model(
    lstm_model,
    train_rnn_loader,
    val_rnn_loader,
    epochs=RNN_EPOCHS,
    lr=RNN_LR,
    patience=RNN_PATIENCE
)

plot_learning_curves(lstm_history, "LSTM")

In [ ]:
lstm_test_metrics, y_true_lstm, y_pred_lstm = evaluate_recurrent(lstm_model, test_rnn_loader, criterion)

print("Métricas en test - LSTM")
for k, v in lstm_test_metrics.items():
    print(f"{k}: {v:.4f}")

print("\nClassification report - LSTM")
print(classification_report(y_true_lstm, y_pred_lstm, target_names=LABEL_NAMES, digits=4))

In [ ]:
recurrent_results = pd.DataFrame([
    {"model": "GRU", **gru_test_metrics},
    {"model": "LSTM", **lstm_test_metrics}
]).sort_values("f1_macro", ascending=False)

display(recurrent_results)

## 7. Preparación para los Transformers

En los Transformers cambia la forma de trabajar con el texto. Ya no se construye un vocabulario propio, porque cada modelo trae su propio tokenizador. Aquí lo importante es aplicar una tokenización coherente, fijar una longitud máxima razonable y hacer fine-tuning sobre el conjunto completo puesto que es lo que más errores y peso de trabajo a causado, la correcta configuración de este apartado.

In [ ]:
hf_train = HFDataset.from_pandas(train_df.reset_index(drop=True))
hf_val = HFDataset.from_pandas(val_df.reset_index(drop=True))
hf_test = HFDataset.from_pandas(test_df.reset_index(drop=True))

hf_datasets = DatasetDict({
    "train": hf_train,
    "validation": hf_val,
    "test": hf_test
})

hf_datasets

In [ ]:
def hf_compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return compute_metrics_np(labels, preds)

def tokenize_for_transformer(examples, tokenizer):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=TRANSFORMER_MAX_LEN
    )

def extract_trainer_history(trainer):
    return pd.DataFrame(trainer.state.log_history)

def plot_transformer_history(logs, title):
    if logs.empty:
        return

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    if "loss" in logs.columns:
        train_logs = logs.dropna(subset=["loss"])
        axes[0].plot(range(1, len(train_logs) + 1), train_logs["loss"], marker="o")
        axes[0].set_title(f"{title} - Train loss")
        axes[0].set_xlabel("Registro")

    if "eval_loss" in logs.columns:
        eval_logs = logs.dropna(subset=["eval_loss"])
        axes[1].plot(range(1, len(eval_logs) + 1), eval_logs["eval_loss"], marker="o", label="eval_loss")
        if "eval_accuracy" in eval_logs.columns:
            axes[1].plot(range(1, len(eval_logs) + 1), eval_logs["eval_accuracy"], marker="o", label="eval_accuracy")
        if "eval_f1_macro" in eval_logs.columns:
            axes[1].plot(range(1, len(eval_logs) + 1), eval_logs["eval_f1_macro"], marker="o", label="eval_f1_macro")
        axes[1].set_title(f"{title} - Validación")
        axes[1].set_xlabel("Evaluación")
        axes[1].legend()

    plt.tight_layout()
    plt.show()

def fine_tune_transformer(model_name, short_name, batch_size):
    print(f"\n[1] Cargando tokenizador: {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        cache_dir=HF_CACHE_DIR
    )

    print("[2] Tokenizando dataset...")
    tokenized = hf_datasets.map(
        lambda batch: tokenize_for_transformer(batch, tokenizer),
        batched=True
    )

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    print(f"[3] Cargando modelo: {model_name}")
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=NUM_CLASSES,
        cache_dir=HF_CACHE_DIR,
        low_cpu_mem_usage=False,
        use_safetensors=False
    )

    output_dir = f"./outputs_{short_name}"

    print("[4] Preparando TrainingArguments...")
    args = TrainingArguments(
        output_dir=output_dir,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="steps",
        logging_steps=200,
        learning_rate=TRANSFORMER_LR,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=TRANSFORMER_EPOCHS,
        weight_decay=TRANSFORMER_WEIGHT_DECAY,
        warmup_ratio=TRANSFORMER_WARMUP_RATIO,
        load_best_model_at_end=True,
        metric_for_best_model="eval_f1_macro",
        greater_is_better=True,
        report_to="none",
        fp16=torch.cuda.is_available(),
        dataloader_num_workers=0,
        save_total_limit=1
    )

    print("[5] Creando Trainer...")
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tokenized["train"],
        eval_dataset=tokenized["validation"],
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=hf_compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=TRANSFORMER_PATIENCE)]
    )

    print("[6] Entrenando...")
    trainer.train()

    print("[7] Evaluando en validación...")
    val_metrics = trainer.evaluate(tokenized["validation"])

    print("[8] Evaluando en test...")
    test_metrics = trainer.evaluate(tokenized["test"])

    logs = extract_trainer_history(trainer)

    print("[9] Fin de fine_tune_transformer()")
    return {
        "trainer": trainer,
        "tokenizer": tokenizer,
        "tokenized": tokenized,
        "logs": logs,
        "val_metrics": val_metrics,
        "test_metrics": test_metrics
    }

## 8. Modelo 3 - DistilBERT

Como primer Transformer se utiliza DistilBERT. Me interesa porque suele mantener bastante bien el rendimiento, pero con menos coste que BERT-base. Es un buen punto intermedio para comparar calidad y tiempo de entrenamiento.

In [ ]:
distilbert_run = fine_tune_transformer(
    model_name=MODEL_3_NAME,
    short_name="distilbert",
    batch_size=DISTIL_BATCH
)

plot_transformer_history(distilbert_run["logs"], "DistilBERT")
print(distilbert_run["test_metrics"])

## 9. Modelo 4 - BERT-base-uncased

Como segundo Transformer se entrena BERT-base-uncased. La idea aquí es ver si una arquitectura más completa mejora de forma clara frente a DistilBERT y si esa mejora compensa el mayor coste computacional.

In [ ]:
bert_run = fine_tune_transformer(
    model_name=MODEL_4_NAME,
    short_name="bert_base",
    batch_size=BERT_BATCH
)

plot_transformer_history(bert_run["logs"], "BERT-base-uncased")
print(bert_run["test_metrics"])

## 10. Comparativa global de modelos

En este punto ya se pueden reunir las métricas de los cuatro modelos para compararlos de forma directa. Así resulta más fácil ver si los Transformers mejoran de verdad a los recurrentes y cuánto.

In [ ]:
all_results = pd.DataFrame([
    {
        "model": "GRU",
        "accuracy": gru_test_metrics["accuracy"],
        "precision_macro": gru_test_metrics["precision_macro"],
        "recall_macro": gru_test_metrics["recall_macro"],
        "f1_macro": gru_test_metrics["f1_macro"]
    },
    {
        "model": "LSTM",
        "accuracy": lstm_test_metrics["accuracy"],
        "precision_macro": lstm_test_metrics["precision_macro"],
        "recall_macro": lstm_test_metrics["recall_macro"],
        "f1_macro": lstm_test_metrics["f1_macro"]
    },
    {
        "model": "DistilBERT",
        "accuracy": distilbert_run["test_metrics"]["eval_accuracy"],
        "precision_macro": distilbert_run["test_metrics"]["eval_precision_macro"],
        "recall_macro": distilbert_run["test_metrics"]["eval_recall_macro"],
        "f1_macro": distilbert_run["test_metrics"]["eval_f1_macro"]
    },
    {
        "model": "BERT-base",
        "accuracy": bert_run["test_metrics"]["eval_accuracy"],
        "precision_macro": bert_run["test_metrics"]["eval_precision_macro"],
        "recall_macro": bert_run["test_metrics"]["eval_recall_macro"],
        "f1_macro": bert_run["test_metrics"]["eval_f1_macro"]
    }
]).sort_values("f1_macro", ascending=False)

display(all_results)

In [ ]:
plt.figure(figsize=(10, 4))
sns.barplot(data=all_results, x="model", y="f1_macro")
plt.title("Comparativa global de modelos (F1 macro en test)")
plt.ylim(0, 1)
plt.xticks(rotation=15)
plt.show()

## 11. Explicabilidad con atención

Para la parte final se analiza el mejor Transformer de los dos entrenados. La idea no es solo enseñar mapas de atención, sino intentar ver si el modelo parece fijarse en palabras o fragmentos que tengan sentido cuando clasifica una noticia.

También interesa mirar ejemplos acertados y fallidos, porque ahí es donde mejor se ve si la atención ayuda de verdad a interpretar lo que está haciendo el modelo.

In [ ]:
best_name = "DistilBERT" if distilbert_run["test_metrics"]["eval_f1_macro"] >= bert_run["test_metrics"]["eval_f1_macro"] else "BERT-base"
best_run = distilbert_run if best_name == "DistilBERT" else bert_run
print("Transformer elegido para atención:", best_name)

In [ ]:
best_trainer = best_run["trainer"]
best_tokenizer = best_run["tokenizer"]
best_tokenized_test = best_run["tokenized"]["test"]

test_pred_output = best_trainer.predict(best_tokenized_test)
test_preds = np.argmax(test_pred_output.predictions, axis=1)

attention_df = test_df.copy()
attention_df["pred"] = test_preds
attention_df["correct"] = (attention_df["label"] == attention_df["pred"]).astype(int)
attention_df["label_name"] = attention_df["label"].map(dict(enumerate(LABEL_NAMES)))
attention_df["pred_name"] = attention_df["pred"].map(dict(enumerate(LABEL_NAMES)))

attention_df.head()

In [ ]:
def select_attention_examples(df, n_correct=5, n_incorrect=5):
    correct_df = df[df["correct"] == 1].groupby("label", group_keys=False).head(max(1, n_correct // NUM_CLASSES + 1))
    incorrect_df = df[df["correct"] == 0].groupby("label", group_keys=False).head(max(1, n_incorrect // NUM_CLASSES + 1))
    out = pd.concat([incorrect_df.head(n_incorrect), correct_df.head(n_correct)], axis=0)
    return out.head(n_correct + n_incorrect).reset_index(drop=True)

examples_df = select_attention_examples(attention_df, n_correct=5, n_incorrect=5)
display(examples_df[["text", "label_name", "pred_name", "correct"]])

In [ ]:
best_model = best_trainer.model.to(device)
best_model.eval()

def get_attention_data(text, tokenizer, model, max_len=TRANSFORMER_MAX_LEN):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_len
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs, output_attentions=True)

    attentions = outputs.attentions
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0].cpu().tolist())

    last_layer = attentions[-1][0].mean(dim=0).cpu().numpy()
    cls_attention = last_layer[0]
    return tokens, last_layer, cls_attention

def plot_cls_attention(text, true_label, pred_label, tokenizer, model):
    tokens, _, cls_attention = get_attention_data(text, tokenizer, model)

    show_n = min(len(tokens), 25)
    tokens = tokens[:show_n]
    cls_attention = cls_attention[:show_n]

    plt.figure(figsize=(12, 3))
    sns.barplot(x=tokens, y=cls_attention)
    plt.xticks(rotation=70)
    plt.title(f"Atención desde [CLS] | real={true_label} | pred={pred_label}")
    plt.tight_layout()
    plt.show()

def plot_last_layer_heatmap(text, true_label, pred_label, tokenizer, model):
    tokens, last_layer, _ = get_attention_data(text, tokenizer, model)

    show_n = min(len(tokens), 20)
    tokens = tokens[:show_n]
    last_layer = last_layer[:show_n, :show_n]

    plt.figure(figsize=(8, 6))
    sns.heatmap(last_layer, xticklabels=tokens, yticklabels=tokens, cmap="viridis")
    plt.xticks(rotation=70)
    plt.yticks(rotation=0)
    plt.title(f"Mapa de atención última capa | real={true_label} | pred={pred_label}")
    plt.tight_layout()
    plt.show()

In [ ]:
for i, row in examples_df.iterrows():
    print(f"\nEjemplo {i+1}")
    print("Clase real :", row["label_name"])
    print("Predicción :", row["pred_name"])
    print("Correcto   :", "Sí" if row["correct"] == 1 else "No")
    print("Texto      :", row["text"][:250], "...")

    plot_cls_attention(
        text=row["text"],
        true_label=row["label_name"],
        pred_label=row["pred_name"],
        tokenizer=best_tokenizer,
        model=best_model
    )

In [ ]:
for i in range(min(3, len(examples_df))):
    row = examples_df.iloc[i]
    plot_last_layer_heatmap(
        text=row["text"],
        true_label=row["label_name"],
        pred_label=row["pred_name"],
        tokenizer=best_tokenizer,
        model=best_model
    )